# PEFT
## 冻结参数

# LoRA
大语言模型是过参数化的（Over-parametrized），它们在针对特定任务进行微调时，权重更新矩阵 
Δ
W
 具有一个很低的“内在秩”（Intrinsic Rank）。<br>
 这意味着，尽管 $ΔW$ 的维度很高，但它所包含的“有效信息”实际上可以被一个远小于其规模的低秩矩阵来表示。对此，LoRA 的核心思想就是用两个更小的“低秩”矩阵 $A$ 和 $B$ 的乘积，来模拟（近似）这个庞大的更新矩阵 $ΔW$。

$$ \Delta W = B \cdot A $$

其中， $W_0 \in \mathbb{R}^{d \times k}$，低秩分解后的 $B \in \mathbb{R}^{d \times r}$， $A \in \mathbb{R}^{r \times k}$，而秩 $r \ll \min(d, k)$。<br>
最终的输出 $h$ 是这两条路径结果的加和：

$$ h = W_0 \cdot x + \Delta W \cdot x = W_0 \cdot x + (B \cdot A) \cdot x $$

在训练时，只有旁路的矩阵 $A$ 和 $B$ 会被更新。通过这种方式，需要优化的参数量就从 $d \times k$ 下降到了 $d \times r + r \times k$。通常，秩 $r$ 会选择一个非常小的值（如 8, 16, 64），使得可训练参数量仅为全量微调的千分之一甚至万分之一。


# AdaLoRA
$$
\Delta W = P \Lambda Q
$$
$$ \bar{I}^{(t)}(w) = \beta_1 \bar{I}^{(t-1)}(w) + (1-\beta_1)I^{(t)}(w) $$

$$ \bar{U}^{(t)}(w) = \beta_2 \bar{U}^{(t-1)}(w) + (1-\beta_2)|I^{(t)}(w) - \bar{I}^{(t)}(w)| $$
$$
S_{k,i} = s(\lambda_{k,i}) + \frac{1}{d_1}\sum_{j=1}^{d_1}s(P_{k,ji}) + \frac{1}{d_2}\sum_{j=1}^{d_2}s(Q_{k,ij})
$$


# QLoRA


In [1]:
import os
import ssl
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CURL_CA_BUNDLE"] = ""  # 禁用 cert 验证
ssl._create_default_https_context = ssl._create_unverified_context

# 然后正常加载模型
from transformers import AutoModelForCausalLM, BitsAndBytesConfig,AutoTokenizer
import torch

model_id = "EleutherAI/pythia-2.8b-deduped"
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # 设置填充符

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

from peft import LoraConfig, get_peft_model
config = LoraConfig(
    r=16,
    target_modules=["query_key_value", "dense"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(model, config)

from datasets import load_dataset
dataset = load_dataset("Abirate/english_quotes")

def tokenize_quotes(batch):
    return tokenizer(batch["quote"],truncation=True)
tokenized_quotes = dataset.map(tokenize_quotes,batched=True)

from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments
peft_model.use_cache = False
train_args = TrainingArguments(
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    max_steps=50,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    output_dir="peft-qlora-english-quotes"
)
quote_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=peft_model,
    train_dataset=tokenized_quotes["train"],
    args=train_args,
    data_collator=quote_collator
)

trainer.train()

peft_model.eval()
peft_model.config.pad_token_id = tokenizer.pad_token_id
prompt = "Be yourself;everyone"
inputs = tokenizer(prompt, return_tensors="pt")
input_ids = inputs["input_ids"].to(peft_model.device)
attention_mask = inputs["attention_mask"].to(peft_model.device)
generation_output = peft_model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_length=50,
    do_sample=True,
    temperature=0.6,
    top_p=0.95,
    top_k=40,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.pad_token_id
)
decoded_output = tokenizer.decode(generation_output[0], skip_special_tokens=True)
print(decoded_output)

d:\deeplearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 388/388 [00:05<00:00, 66.86it/s, Materializing param=gpt_neox.layers.31.post_attention_layernorm.weight]  
d:\deeplearning\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
d:\deeplearning\.venv\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.fl

Step,Training Loss
1,2.598916
2,2.626944
3,2.183646
4,2.670139
5,2.318218
6,2.701851
7,2.623686
8,2.418115
9,2.508394
10,2.850463


d:\deeplearning\.venv\Lib\site-packages\bitsandbytes\autograd\_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Be yourself;everyone else is taken.”If you are always trying to impress others,you will never live a full life. If you aren't interested in people then why should they be interested in you? As humans we have an innate desire for


In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # ✅ 严格无空格

from datasets import load_dataset

# 直接加载，不手动 monkey-patch requests 或 ssl
dataset = load_dataset("Abirate/english_quotes")
print(dataset["train"][0])

d:\deeplearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\deeplearning\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\zhuxu\.cache\huggingface\hub\datasets--Abirate--english_quotes. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this arti

{'quote': '“Be yourself; everyone else is already taken.”', 'author': 'Oscar Wilde', 'tags': ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']}
